# 04 - CLAM-SB training, instance-loss ablation and attention analysis

**Owner:** Person 2 (CLAM-SB)
**Goal:** test whether forcing the model to distinguish high- and low-attention
patches improves weakly supervised ISUP grading.

This notebook is a thin front-end over the modules in `src/`. Everything it
runs is also available as a one-line shell command:

```bash
bash scripts/run_mock_demo.sh          # synthetic data, works today
bash scripts/run_person2_experiments.sh # real PANDA features, once Person 1 delivers
```

> If `manifest = data/mock/manifest.csv` the numbers below are **synthetic** and
> must never be reported. Switch `MANIFEST`/`FEATURES` to the real paths for the
> reportable run.

## 0. Setup

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch

from src.dataset import class_distribution, describe_dataset, read_manifest, split_records
from src.models import build_model, count_parameters

print("torch", torch.__version__, "| device:", "cuda" if torch.cuda.is_available() else "cpu")

# ---- paths: swap these three lines for the real PANDA run -------------------
MANIFEST = ROOT / "data/mock/manifest.csv" if (ROOT / "data/mock/manifest.csv").exists() else ROOT / "data/common_split.csv"
FEATURES = MANIFEST.parent / "features"
CONFIG   = ROOT / "configs/clam_sb_mock.yaml" if "mock" in str(MANIFEST) else ROOT / "configs/clam_sb.yaml"
OUT      = ROOT / ("results/mock_clam_sb" if "mock" in str(MANIFEST) else "results/metrics/clam_sb")
SYNTHETIC = "mock" in str(MANIFEST)

print("manifest:", MANIFEST)
print("features:", FEATURES)
print("config  :", CONFIG)
print("SYNTHETIC (not reportable):", SYNTHETIC)

## 1. Inspect the data contract

Every slide must have `features [N, D]`, ideally with `coords [N, 2]` for heatmaps.

In [ ]:
summary = describe_dataset(MANIFEST, FEATURES)
print(json.dumps({k: v for k, v in summary.items() if k != "class_distribution"}, indent=2))

records = read_manifest(MANIFEST)
dist = class_distribution(records)
pd.DataFrame(dist).T.fillna(0).astype(int)

## 2. Model sanity check

Shapes, instance-loss value and parameter count before spending any compute.

In [ ]:
import yaml
cfg = yaml.safe_load(CONFIG.read_text())
model_cfg = dict(cfg["model"])
model_cfg["in_dim"] = summary["feature_dim"] or model_cfg["in_dim"]

torch.manual_seed(0)
model = build_model("clam_sb", model_cfg)
n_patches = 64
features = torch.randn(n_patches, model_cfg["in_dim"])
labels = 2

out = model(features, labels=labels)
print("logits      ", tuple(out["logits"].shape))
print("attention   ", tuple(out["attention"].shape), "sum =", float(out["attention"].sum()))
print("instance loss", float(out["instance_loss"]))
print("top-k idx   ", out["top_indices"].tolist())
print("parameters  ", f"{count_parameters(model):,}")

# The ablation must change only the loss, never the architecture:
model_ablation = build_model("clam_sb", {**model_cfg, "instance_loss_weight": 0.0})
print("same parameter count in both ablation arms:", count_parameters(model) == count_parameters(model_ablation))

## 3. Train the ablation: with vs without instance-level loss

Identical features, split, seeds and architecture; only `instance_loss_weight`
changes (0.30 vs 0.0). Validation QWK drives early stopping; the test split is
touched once, at the end.

In [ ]:
n_seeds = 1 if SYNTHETIC else 3
cmd = [
    sys.executable, "-m", "src.train_clam_sb",
    "--manifest", str(MANIFEST),
    "--features-root", str(FEATURES),
    "--config", str(CONFIG),
    "--out-dir", str(OUT),
    "--seeds", *[str(s) for s in range(n_seeds)],
]
print(" ".join(cmd))
result = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
print(result.stdout[-2500:])
print(result.stderr[-1500:])
result.returncode

## 4. Ablation table

In [ ]:
ablation = json.loads((OUT / "ablation_clam_sb.json").read_text())
rows = []
for mode, agg in ablation.items():
    rows.append({
        "setting": mode,
        "QWK mean": agg.get("qwk", {}).get("mean"),
        "QWK sd": agg.get("qwk", {}).get("std"),
        "accuracy": agg.get("accuracy", {}).get("mean"),
        "balanced acc": agg.get("balanced_accuracy", {}).get("mean"),
        "macro F1": agg.get("macro_f1", {}).get("mean"),
    })
df = pd.DataFrame(rows).round(4)
print((OUT / "ablation_clam_sb.md").read_text())
df

## 5. Did instance supervision sharpen attention?

Beyond QWK, CLAM claims sharper attention. We measure it with the attention
entropy (in nats, and normalised by `log N` so slides with different patch
counts are comparable) and the share of attention mass held by the top-10
patches. Lower entropy / higher top-10 share = more focused.

In [ ]:
eval_cmd = [
    sys.executable, "-m", "src.evaluate_clam_sb",
    "--checkpoint", str(OUT / "with_instance/checkpoints/clam_sb_seed0_best.pt"),
    "--manifest", str(MANIFEST), "--features-root", str(FEATURES),
    "--out-dir", str(OUT / "eval_with_instance"), "--split", "all",
]
eval_result = subprocess.run(eval_cmd, cwd=ROOT, capture_output=True, text=True)
print(eval_result.returncode, eval_result.stdout[-600:])

eval_cmd[eval_cmd.index("--checkpoint") + 1] = str(OUT / "without_instance/checkpoints/clam_sb_seed0_best.pt")
eval_cmd[eval_cmd.index("--out-dir") + 1] = str(OUT / "eval_without_instance")
subprocess.run(eval_cmd, cwd=ROOT, capture_output=True, text=True)

att_with = pd.read_csv(OUT / "eval_with_instance/attention_clam_sb_seed0.csv")
att_without = pd.read_csv(OUT / "eval_without_instance/attention_clam_sb_seed0.csv")

attention_summary = pd.DataFrame({
    "with instance loss": [att_with["attention_entropy"].mean(), att_with["normalised_entropy"].mean(), att_with["top10_attention_share"].mean()],
    "without instance loss": [att_without["attention_entropy"].mean(), att_without["normalised_entropy"].mean(), att_without["top10_attention_share"].mean()],
}, index=["attention entropy", "normalised entropy", "top-10 attention share"]).round(4)
attention_summary

## 6. Heatmaps and localisation overlap

Attention overlays for the test slides, plus (when masks exist) a quantitative overlap score at the top-10/20/30% of patches.

In [ ]:
from src.export_clam_attention import export_attention_heatmaps

heatmap_dir = ROOT / ("results/mock_heatmaps_nb" if SYNTHETIC else "results/heatmaps/clam_sb_nb")
thumbs = MANIFEST.parent / "thumbnails"
masks = MANIFEST.parent / "masks"

summary_h = export_attention_heatmaps(
    checkpoint=OUT / "with_instance/checkpoints/clam_sb_seed0_best.pt",
    manifest=MANIFEST, features_root=FEATURES, out_dir=heatmap_dir, split="test",
    thumbnails=thumbs if thumbs.exists() else None,
    masks=masks if masks.exists() else None,
)
print(json.dumps(summary_h, indent=2)[:1200])

In [ ]:
from IPython.display import Image, display
overlays = sorted((heatmap_dir / "images").glob("*_overlay.png")) or sorted((heatmap_dir / "images").glob("*_attention.png"))
for path in overlays[:3]:
    print(path.name)
    display(Image(filename=str(path), width=480))
if (heatmap_dir / "localisation_scores.csv").exists():
    display(pd.read_csv(heatmap_dir / "localisation_scores.csv").head(10))

## 7. Paired comparison against the ABMIL baseline

Both models on the same 60 test slides: paired bootstrap CI for the QWK difference plus an exact McNemar test.

In [ ]:
abmil_csv = ROOT / ("results/mock_abmil/predictions_abmil_seed0.csv" if SYNTHETIC else "results/metrics/abmil/predictions_abmil_seed0.csv")
score = {
    "qwk": ablation.get("with_instance", {}).get("qwk", {}).get("mean"),
    "without": ablation.get("without_instance", {}).get("qwk", {}).get("mean"),
}
print("QWK with instance loss   :", score["qwk"])
print("QWK without instance loss:", score["without"])

if abmil_csv.exists():
    from src.compare_clam_abmil import compare
    comparison = compare(
        abmil=abmil_csv,
        clam=OUT / "with_instance/predictions_clam_sb_seed0.csv",
        out_dir=ROOT / "results/comparison_nb",
        abmil_label="ABMIL", clam_label="CLAM-SB + instance loss",
    )
    print((ROOT / "results/comparison_nb/comparison.md").read_text())
else:
    print("ABMIL predictions not found -- Person 1 has not produced them yet:", abmil_csv)

## 8. Reporting checklist (fill this in before the write-up)

- [ ] Real PANDA features and `data/common_split.csv` used (not `data/mock`).
- [ ] 3 seeds trained; QWK reported as mean +/- sd with a bootstrap 95% CI.
- [ ] Ablation table (with vs without instance loss) copied from `ablation_clam_sb.md`.
- [ ] Paired ABMIL vs CLAM-SB comparison with CI and McNemar p-value.
- [ ] Attention sharpness numbers (entropy / top-10 share) for both arms.
- [ ] Heatmaps for >= 10 test slides, with the localisation caveat stated.
- [ ] The headline sentence: *CLAM-SB tests whether forcing the model to
      distinguish high- and low-attention patches improves weakly supervised
      grading.*
- [ ] Limitations: research prototype, not clinical; 420-slide PANDA subset may
      not generalise; ISUP imbalance makes accuracy alone misleading; attention
      is not a tumour segmentation; external validation required.